# Harmonising Dft Datasets (transport accessibility features) from 2011 to 2021.
Dataset harmonisation was necessary because the datasets used in the study were published at different times and based on different geographical boundaries. In particular, the DfT Journey Time Statistics used 2011 LSOA geography, while the Census and contextual datasets were based on 2021 LSOAs. Harmonising all spatial datasets to a common 2021 LSOA geography ensured that variables referred to comparable geographical units, allowing them to be reliably integrated into a single master dataset and enabling consistent spatial comparison, rural-urban analysis, and machine-learning modelling across England.

In [2]:
import pandas as pd
import numpy as np

lookup = pd.read_excel(
    "LSOA_2011to2021.xlsx"
)

In [ ]:
employment_centres = pd.read_excel("DFT Datasets/employment_features.xlsx")
food_store = pd.read_excel("DFT Datasets/food_store_features.xlsx")
gp = pd.read_excel("DFT Datasets/gp_accessibility_features.xlsx")
hospital = pd.read_excel("DFT Datasets/hospital_accessibility_features.xlsx")
town_centre = pd.read_excel("DFT Datasets/town_centre_features.xlsx")

In [6]:
#Checking datasets for errors
datasets = {
    "Employment": employment_centres,
    "Food Store": food_store,
    "GP": gp,
    "Hospital": hospital,
    "Town Centre": town_centre
}

for name, df in datasets.items():
    print(
        name,
        "| rows:", len(df),
        "| unique LSOAs:", df["LSOA_code"].nunique(),
        "| duplicates:", df["LSOA_code"].duplicated().sum()
    )

Employment | rows: 32844 | unique LSOAs: 32844 | duplicates: 0
Food Store | rows: 32844 | unique LSOAs: 32844 | duplicates: 0
GP | rows: 32844 | unique LSOAs: 32844 | duplicates: 0
Hospital | rows: 32844 | unique LSOAs: 32844 | duplicates: 0
Town Centre | rows: 32844 | unique LSOAs: 32844 | duplicates: 0


In [7]:
#harmonisation function
def harmonise_lsoa(df, lookup):
    
    # Add 2021 LSOA geography
    df_harmonised = df.merge(
        lookup,
        left_on="LSOA_code",
        right_on="LSOA11CD",
        how="left",
        validate="one_to_one"
    )

    # Check whether any DfT LSOAs failed to match
    unmatched = df_harmonised["LSOA21CD"].isna().sum()
    print("Unmatched LSOAs:", unmatched)

    # Remove lookup's duplicate copy of the 2011 code
    df_harmonised = df_harmonised.drop(columns=["LSOA11CD"])

    return df_harmonised

In [8]:
employment_21 = harmonise_lsoa(employment_centres, lookup)
food_store_21 = harmonise_lsoa(food_store, lookup)
gp_21 = harmonise_lsoa(gp, lookup)
hospital_21 = harmonise_lsoa(hospital, lookup)
town_centre_21 = harmonise_lsoa(town_centre, lookup)

Unmatched LSOAs: 0
Unmatched LSOAs: 0
Unmatched LSOAs: 0
Unmatched LSOAs: 0
Unmatched LSOAs: 0


In [9]:
mapping_counts = (
    employment_21
    .groupby("LSOA21CD")
    .size()
    .value_counts()
    .sort_index()
)

print(mapping_counts)

1    32650
2       97
Name: count, dtype: int64


In [10]:
def aggregate_to_2021(df):

    # Identify numeric feature columns
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

    # Aggregate numeric accessibility features
    df_2021 = (
        df
        .groupby(["LSOA21CD", "LSOA21NM"], as_index=False)[numeric_cols]
        .mean()
    )

    return df_2021

In [11]:
employment_21 = aggregate_to_2021(employment_21)
food_store_21 = aggregate_to_2021(food_store_21)
hospital_21 = aggregate_to_2021(hospital_21)
town_centre_21 = aggregate_to_2021(town_centre_21)

In [13]:
gp_21 = ( gp_21.groupby(["LSOA21CD", "LSOA21NM"], as_index=False)
    .agg({
        "GP_pop": "sum",
        "GPPTt": "mean",
        "GPCart": "mean",
        "GPCyct": "mean",
        "GPWalkt": "mean",
        "GPPT30pct": "mean",
        "GP_PT_Car_Gap": "mean"
    })
)

In [14]:
#verifying
datasets_2021 = {
    "Employment": employment_21,
    "Food Store": food_store_21,
    "GP": gp_21,
    "Hospital": hospital_21,
    "Town Centre": town_centre_21
}

for name, df in datasets_2021.items():
    print(
        name,
        "| rows:", len(df),
        "| unique 2021 LSOAs:", df["LSOA21CD"].nunique(),
        "| duplicates:", df["LSOA21CD"].duplicated().sum()
    )

Employment | rows: 32747 | unique 2021 LSOAs: 32747 | duplicates: 0
Food Store | rows: 32747 | unique 2021 LSOAs: 32747 | duplicates: 0
GP | rows: 32747 | unique 2021 LSOAs: 32747 | duplicates: 0
Hospital | rows: 32747 | unique 2021 LSOAs: 32747 | duplicates: 0
Town Centre | rows: 32747 | unique 2021 LSOAs: 32747 | duplicates: 0


All the Dft features that correspond to transport accessbility to essental services were harmonised to 2021LSOA census. The features were cleaned and then added to the following datasets.

In [15]:
employment_21.to_excel(
    "employmentcentres_features2021.xlsx",
    index=False
)

food_store_21.to_excel(
    "food_store_features2021.xlsx",
    index=False
)

gp_21.to_excel(
    "gp_accessibility_features2021.xlsx",
    index=False
)

hospital_21.to_excel(
    "hospital_accessibility_features2021.xlsx",
    index=False
)

town_centre_21.to_excel(
    "town_centre_features2021.xlsx",
    index=False
)

These datasets will be used as inputs in Master notebook for building the final integrated data for further feature and target(Accessibility Inequality Metric) construction